## Test Alamanc Generation 

In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import os

# my modules
from src.keplarian_ephemeris import *
from src.orbit_manager import *

In [ ]:
basedir = "/Users/keidaiiiyama/Documents/sw_navlab/LuPNT-private/output/Ephemeris/"
orbm_save_dir = basedir + "data/orbits"
figdir = basedir + "figures/orbit_analysis"

if not os.path.exists(figdir):
    os.makedirs(figdir)

In [ ]:
from src.almanac import initialize_orbit, setup_orbit

orbit = "LCRNS"  # Choose from "LCRNS", "Polar", "NRHO"
# orbit = "Moonlight"
# orbit = "LNSS"
# orbit = "Polar"

t0_tai, rv0_mci = initialize_orbit(orbit, svid=0)

long_term_analysis = False

if long_term_analysis:
    if orbit == "NRHO":
        t_days = 60
    else:
        t_days = 365
    figname_coe = os.path.join(figdir, f"{orbit}_coe.pdf")

else:
    t_days = 15
    figname_coe = None

In [ ]:
results = setup_orbit(
    t0_tai, rv0_mci, t_days=t_days, plot_cart=True, plot_coe=True, figname=figname_coe
)

## Fourier Analysis

In [ ]:
t_tai = results["t_tai"]
rvbf = results["rv_prop_pa"]
rvbf_w = results["rv_prop_pa_w"]
coe = results["coe_prop_pa"]

sma = coe[:, 0]
ecc = coe[:, 1]
inc = coe[:, 2]
raan = coe[:, 3]
argp = coe[:, 4]
M = coe[:, 5]

# Fourier analysis
from scipy.fft import fft, ifft, fftfreq

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes = axes.ravel()

Torb = 2 * np.pi * np.sqrt(sma**3 / pnt.GM_MOON)  # Orbital period [s]
print(f"Orbital Period: {np.mean(Torb)/3600:.2f} hours")

labels = [
    "Semi-major Axis",
    "Eccentricity",
    "Inclination",
    "Longitude of Ascending Node",
    "Argument of Periapsis",
    "Mean Anomaly",
]

plot_inv = 10

for i in range(5):
    # Do FFT
    N = len(t_tai)
    T = (t_tai[-1] - t_tai[0]) / N
    yf = fft(coe[:, i])[: N // 2]
    xf = fftfreq(N, T)[: N // 2]

    xf = np.hstack([xf[:100], xf[100::plot_inv]])
    yf = np.hstack([yf[:100], yf[100::plot_inv]])

    axes[i].set_ylabel("Amplitude", fontsize=14)
    axes[i].set_xlabel("Frequency [1/days]", fontsize=14)
    axes[i].set_title(labels[i], fontsize=18, fontweight="bold")
    axes[i].tick_params(axis="both", which="major", labelsize=12)
    # Plot time series
    axes[i].plot(xf * 86400, 2.0 / N * np.abs(yf), "o-")
    axes[i].grid()
    # axes[i].set_xlim(0, 50)
    axes[i].axvline(x=1 / 27.3, color="red", linestyle="--", label="Lunar Period")
    axes[i].axvline(
        x=1 / 13.65, color="orange", linestyle="--", label="Half Lunar Period"
    )
    axes[i].axvline(x=1 / 9.1, color="green", linestyle="--", label="1/3 Lunar Period")
    axes[i].axvline(
        x=1 / (Torb.mean() / 86400),
        color="purple",
        linestyle="--",
        label="Orbital Period",
    )
    axes[i].set_yscale("log")
    axes[i].set_xscale("log")
    if i == 4:
        axes[i].legend()

plt.tight_layout()
if long_term_analysis:
    plt.savefig(figdir + f"/{orbit}_coe_fft.pdf", dpi=300)
plt.show()

## Test with Cartesian Ephemeris (Does not Work)

In [ ]:
from src.cartesian_ephemeris import CartesianEphemeris

# First guess: just fit cartesian ephemeris
t_tai = results["t_tai"]
rvbf = results["rv_prop_pa"]
rvbf_w = results["rv_prop_pa_w"]
tspan = t_tai - t_tai[0]

cart_ephem = CartesianEphemeris(
    order=30,
    use_kep=True,
    use_rsw=False,
    use_fourier=True,
    use_meq=False,
    poly_type="chebyshev",
    convert_to_coe=False,
    body=pnt.MOON,
    print_info=False,
)

ephem_x = cart_ephem.fit(tspan, rvbf, fit_obj="lsq")

diff_xyz, _, _ = cart_ephem.eval_fit_error(
    tspan,
    rvbf,
    rvbf_w,
    ephem_x,
    use_grad_for_velfit=True,
    print_stats=True,
    use_rtn=False,
)

# cart_ephem.plot_fit_error(tspan, rvbf, rvbf_w, ephem_x, plot_init=False, print_stats=True,
#         plot_diff=True, use_grad_for_velfit=True, ylim_pos=None, ylim_vel=None, use_rtn=False, t_data_fit=None
#     )

In [ ]:
cart_ephem.plot_fit_error(
    tspan,
    rvbf,
    rvbf_w,
    ephem_x,
    plot_init=False,
    print_stats=True,
    plot_diff=True,
    use_grad_for_velfit=True,
    ylim_pos=None,
    ylim_vel=None,
    use_rtn=True,
    t_data_fit=None,
    in_kms=True,
    plot_azel=True,
)

## Test with Keplarian Ephemeris

In [ ]:
almfig_dir = basedir + "/figures/almanac_fits"
if not os.path.exists(almfig_dir):
    os.makedirs(almfig_dir)

In [ ]:
from src.almanac import Almanac

ptype = "chebyshev"

kep_dict = {
    # 0 (constant), 1 (t), 2 (t^2), 3(t^3), 4(t^4), sin-cos
    "a": {"linear": 2, "fourier": 1, "fourier_sidereal": 0, "polytype": ptype},
    "e": {"linear": 1, "fourier": 0, "fourier_sidereal": 1, "polytype": ptype},
    "w": {"linear": -1, "fourier": 0, "fourier_sidereal": 0, "polytype": ptype},
    "M": {"linear": 1, "fourier": 0, "fourier_sidereal": 1, "polytype": ptype},
    "i": {"linear": 1, "fourier": 0, "fourier_sidereal": 1, "polytype": ptype},
    "l": {"linear": 1, "fourier": 0, "fourier_sidereal": 1, "polytype": ptype},
    "u": {"linear": 1, "fourier": 0, "fourier_sidereal": 1, "polytype": ptype},
}

# if orbit == "LCRNS":
#     use_full_sidereal = False
# elif orbit == "Polar":
#     use_full_sidereal = False
# else:
#     use_full_sidereal = False

print("tspan shape:", tspan.shape)
print("rvbf shape:", rvbf.shape)

alm = Almanac(
    kep_dict, body=pnt.MOON, use_sma_special=False, print_info=True, use_op=False
)
alm_x = alm.fit(
    tspan,
    rvbf,
    fit_obj="lsq",
    print_result=True,
    plot_fit=True,
    figname=os.path.join(almfig_dir, f"{orbit}_almanac"),
)

In [ ]:
alm.plot_fit_error(
    tspan,
    rvbf,
    rvbf_w,
    alm_x,
    plot_init=False,
    print_stats=True,
    plot_diff=True,
    use_grad_for_velfit=True,
    ylim_pos=None,
    ylim_vel=None,
    use_rtn=True,
    t_data_fit=None,
    in_kms=True,
    plot_azel=True,
    figname_error=os.path.join(almfig_dir, f"{orbit}_almanac_fit_error.pdf"),
    figname_azel=os.path.join(almfig_dir, f"{orbit}_almanac_azel.pdf"),
    azel_xlim=10,
    mask_southpole=True,
    plot_prctile=True,
    plot_azel_legend=True,
    tworows_azel=True,
    plot_azel_range_doppler=True,
    figname_azel_range_doppler=os.path.join(
        almfig_dir, f"{orbit}_almanac_azel_range_doppler.pdf"
    ),
)